In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import random
from PIL import Image
import matplotlib.pyplot as plt

# ✅ Dataset path
base_path = "/content/drive/MyDrive/Crops Dtaset/Agricultural-crops"

# ✅ Dictionary to store class and images
classes = {}
for root, dirs, files in os.walk(base_path):
    # Ignore the root folder itself
    if root == base_path:
        continue
    class_name = os.path.basename(root)
    image_files = [os.path.join(root, f) for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if image_files:
        classes[class_name] = image_files

# ✅ Summary
if not classes:
    print("❌ No image classes found. Check your dataset path.")
else:
    print(f"✅ Total Classes Found: {len(classes)}\n")
    total_imgs = 0
    for cls, imgs in classes.items():
        print(f"📁 {cls}: {len(imgs)} images")
        total_imgs += len(imgs)
    print(f"\n🌿 Total Images in Dataset: {total_imgs}")

    # ✅ Display a few random samples from each class
    for cls, imgs in classes.items():
        print(f"\n🔹 Showing samples from class: {cls}")
        num = min(5, len(imgs))
        samples = random.sample(imgs, num)

        plt.figure(figsize=(10, 4))
        for i, img_path in enumerate(samples):
            img = Image.open(img_path)
            plt.subplot(1, num, i + 1)
            plt.imshow(img)
            plt.axis('off')
            plt.title(cls, fontsize=8)
        plt.tight_layout()
        plt.show()


In [ ]:
# ============================================================
# 🌾 Crop Classification using LeNet (Simplified)
# Author: Muhammad Umar (2025)
# ============================================================

!pip install torch torchvision tqdm matplotlib --quiet

import os, time, copy, random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader
from torchvision import transforms, datasets

# ----------------- CONFIG -----------------
DATA_DIR = "/content/drive/MyDrive/Crops Dtaset/Agricultural-crops"  # your dataset root
OUT_DIR = "/content/drive/MyDrive/cv_training_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("📦 Device:", DEVICE)

# ----------------- DATA -----------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
num_classes = len(dataset.classes)
print(f"✅ Found {len(dataset)} images across {num_classes} classes: {dataset.classes}")

# split data
val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
train_ds.dataset.transform = train_transform
val_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ----------------- MODEL (LeNet) -----------------
class LeNet(nn.Module):
    def __init__(self, nc=3, classes=num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(nc, 6, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(6, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4,4)),
            nn.Flatten(),
            nn.Linear(16*4*4, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, classes)
        )
    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = LeNet().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ----------------- TRAIN / VALIDATION -----------------
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    # validation
    model.eval()
    vloss, vcorrect, vtotal = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            vloss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            vcorrect += (preds == labels).sum().item()
            vtotal += imgs.size(0)
    val_loss = vloss / vtotal
    val_acc = vcorrect / vtotal

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"📊 Epoch {epoch}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Loss: {val_loss:.4f}")

torch.save(model.state_dict(), os.path.join(OUT_DIR, "lenet_final.pth"))
print("✅ Training finished. Model saved.")

# ----------------- PLOT RESULTS -----------------
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend(); plt.title("Loss Curve")

plt.subplot(1,2,2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.legend(); plt.title("Accuracy Curve")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "lenet_training_curves.png"))
plt.show()

print("📦 Outputs saved to:", OUT_DIR)


In [ ]:
# ============================================================
# 🌾 Crop Classification using AlexNet (Simplified Fine-Tuning)
# Author: Muhammad Umar (2025)
# ============================================================

!pip install torch torchvision tqdm matplotlib --quiet

import os, time, copy, random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader
from torchvision import transforms, datasets, models

# ----------------- CONFIG -----------------
DATA_DIR = "/content/drive/MyDrive/Crops Dtaset/Agricultural-crops"  # your dataset root
OUT_DIR = "/content/drive/MyDrive/cv_training_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("📦 Device:", DEVICE)

# ----------------- DATA -----------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
num_classes = len(dataset.classes)
print(f"✅ Found {len(dataset)} images across {num_classes} classes: {dataset.classes}")

# split data
val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
train_ds.dataset.transform = train_transform
val_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ----------------- MODEL (AlexNet) -----------------
# Load pretrained AlexNet from torchvision
model = models.alexnet(pretrained=True)

# Replace final layer for our crop dataset
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

# (Optional) freeze feature extractor if dataset small
# for param in model.features.parameters():
#     param.requires_grad = False

model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

# ----------------- TRAIN / VALIDATION -----------------
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    # validation
    model.eval()
    vloss, vcorrect, vtotal = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            vloss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            vcorrect += (preds == labels).sum().item()
            vtotal += imgs.size(0)
    val_loss = vloss / vtotal
    val_acc = vcorrect / vtotal

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"📊 Epoch {epoch}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Loss: {val_loss:.4f}")

torch.save(model.state_dict(), os.path.join(OUT_DIR, "alexnet_final.pth"))
print("✅ Training finished. Model saved.")

# ----------------- PLOT RESULTS -----------------
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend(); plt.title("Loss Curve")

plt.subplot(1,2,2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.legend(); plt.title("Accuracy Curve")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "alexnet_training_curves.png"))
plt.show()

print("📦 Outputs saved to:", OUT_DIR)


In [ ]:
# ============================================================
# 🌾 Crop Classification using AlexNet (Simplified Fine-Tuning)
# Author: Muhammad Umar (2025)
# ============================================================

!pip install torch torchvision tqdm matplotlib --quiet

import os, time, copy, random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader
from torchvision import transforms, datasets, models

# ----------------- CONFIG -----------------
DATA_DIR = "/content/drive/MyDrive/Crops Dtaset/Agricultural-crops"  # your dataset root
OUT_DIR = "/content/drive/MyDrive/cv_training_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("📦 Device:", DEVICE)

# ----------------- DATA -----------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
num_classes = len(dataset.classes)
print(f"✅ Found {len(dataset)} images across {num_classes} classes: {dataset.classes}")

# split data
val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
train_ds.dataset.transform = train_transform
val_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ----------------- MODEL (AlexNet) -----------------
# Load pretrained AlexNet from torchvision
model = models.alexnet(pretrained=True)

# Replace final layer for our crop dataset
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

# (Optional) freeze feature extractor if dataset small
# for param in model.features.parameters():
#     param.requires_grad = False

model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

# ----------------- TRAIN / VALIDATION -----------------
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    # validation
    model.eval()
    vloss, vcorrect, vtotal = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            vloss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            vcorrect += (preds == labels).sum().item()
            vtotal += imgs.size(0)
    val_loss = vloss / vtotal
    val_acc = vcorrect / vtotal

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"📊 Epoch {epoch}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Loss: {val_loss:.4f}")

torch.save(model.state_dict(), os.path.join(OUT_DIR, "alexnet_final.pth"))
print("✅ Training finished. Model saved.")

# ----------------- PLOT RESULTS -----------------
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend(); plt.title("Loss Curve")

plt.subplot(1,2,2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.legend(); plt.title("Accuracy Curve")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "alexnet_training_curves.png"))
plt.show()

print("📦 Outputs saved to:", OUT_DIR)


In [ ]:
# ============================================================
# 🌾 Crop Classification using DenseNet121 (Transfer Learning)
# Author: Muhammad Umar (2025)
# ============================================================

!pip install torch torchvision tqdm matplotlib --quiet

import os, torch, random, numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from torch import nn, optim
from tqdm import tqdm

# ---------------- CONFIG ----------------
DATA_DIR = "/content/drive/MyDrive/Crops Dtaset/Agricultural-crops"
OUT_DIR = "/content/drive/MyDrive/cv_training_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
IMAGE_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("📦 Device:", DEVICE)

# ---------------- DATA ----------------
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=train_tf)
num_classes = len(dataset.classes)
print(f"✅ Found {len(dataset)} images across {num_classes} classes")

val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])
train_ds.dataset.transform = train_tf
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ---------------- MODEL ----------------
model = models.densenet121(pretrained=True)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, num_classes)
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ---------------- TRAINING ----------------
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

    val_loss, val_correct, val_total = 0, 0, 0
    model.eval()
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += imgs.size(0)

    print(f"📊 Epoch {epoch} | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}")

torch.save(model.state_dict(), os.path.join(OUT_DIR, "densenet121_crops.pth"))
print("✅ Model saved successfully!")


In [ ]:
# ============================================================
# 🌾 Crop Disease Detection (VGG16 / MobileNet)
# Author: Muhammad Umar (2025)
# ============================================================

!pip install torch torchvision tqdm matplotlib --quiet

import os, random, numpy as np, matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# ---------------- CONFIG ----------------
DATA_DIR = "/content/drive/MyDrive/Crops Dtaset/Agricultural-crops"
OUT_DIR = "/content/drive/MyDrive/Crop_Disease_Detection/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_TYPE = "mobilenet"   # 🔁 Change to "vgg16" or "mobilenet"
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-4
IMAGE_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📦 Using {DEVICE} and model: {MODEL_TYPE.upper()}")

# ---------------- DATA ----------------
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=train_tf)
num_classes = len(dataset.classes)
print(f"✅ Found {len(dataset)} images across {num_classes} classes: {dataset.classes}")

val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])
train_ds.dataset.transform = train_tf
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ---------------- MODEL ----------------
if MODEL_TYPE == "vgg16":
    model = models.vgg16(pretrained=True)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
elif MODEL_TYPE == "mobilenet":
    model = models.mobilenet_v2(pretrained=True)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
else:
    raise ValueError("❌ MODEL_TYPE must be 'vgg16' or 'mobilenet'")

model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ---------------- TRAINING ----------------
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    vloss, vcorrect, vtotal = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            vloss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            vcorrect += (preds == labels).sum().item()
            vtotal += imgs.size(0)

    val_loss = vloss / vtotal
    val_acc = vcorrect / vtotal

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"📊 Epoch {epoch}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# ---------------- SAVE MODEL ----------------
torch.save(model.state_dict(), os.path.join(OUT_DIR, f"{MODEL_TYPE}_crops.pth"))
print(f"✅ Model saved: {MODEL_TYPE}_crops.pth")

# ---------------- PLOT RESULTS ----------------
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend(); plt.title("Loss Curve")

plt.subplot(1,2,2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.legend(); plt.title("Accuracy Curve")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{MODEL_TYPE}_training_curves.png"))
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
import random

# ✅ Load Trained Model
model.eval()

# Function to show sample validation results
def show_validation_predictions(model, val_loader, classes, num_samples=6):
    model.eval()
    imgs, labels = next(iter(val_loader))
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    outputs = model(imgs)
    _, preds = torch.max(outputs, 1)

    plt.figure(figsize=(12,6))
    for i in range(num_samples):
        img = imgs[i].permute(1,2,0).cpu().numpy()
        img = np.clip((img * np.array([0.229,0.224,0.225])) + np.array([0.485,0.456,0.406]), 0, 1)
        plt.subplot(2, num_samples//2, i+1)
        plt.imshow(img)
        plt.title(f"Pred: {classes[preds[i]]}\nTrue: {classes[labels[i]]}",
                  color=("green" if preds[i]==labels[i] else "red"))
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# 🔹 Run the visual validation check
show_validation_predictions(model, val_loader, dataset.classes)


In [ ]:
# ============================================================
# 🎥 Visual + 3D Conversion (MiDaS Depth Map) for Predictions
# Works after model training (like MobileNet)
# ============================================================

import matplotlib.pyplot as plt
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
import random

# ✅ Load MiDaS for 3D depth estimation
from transformers import DPTForDepthEstimation, DPTImageProcessor

depth_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-large")
depth_processor = DPTImageProcessor.from_pretrained("Intel/dpt-large")

# ✅ Set model to evaluation mode
model.eval()

# Function to show predictions + depth maps
def show_validation_predictions_with_3D(model, val_loader, classes, num_samples=4):
    model.eval()
    imgs, labels = next(iter(val_loader))
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    outputs = model(imgs)
    _, preds = torch.max(outputs, 1)

    plt.figure(figsize=(14, 8))
    for i in range(num_samples):
        img = imgs[i].permute(1,2,0).cpu().numpy()
        img = np.clip((img * np.array([0.229,0.224,0.225])) + np.array([0.485,0.456,0.406]), 0, 1)

        # --- Generate 3D depth map using MiDaS ---
        pil_img = Image.fromarray((img * 255).astype(np.uint8))
        inputs = depth_processor(images=pil_img, return_tensors="pt")
        with torch.no_grad():
            predicted_depth = depth_model(**inputs).predicted_depth
        depth = predicted_depth.squeeze().cpu().numpy()

        # --- Normalize depth for visualization ---
        depth = (depth - depth.min()) / (depth.max() - depth.min())

        # --- Display side by side ---
        plt.subplot(2, num_samples, i + 1)
        plt.imshow(img)
        plt.title(f"Pred: {classes[preds[i]]}\nTrue: {classes[labels[i]]}",
                  color=("green" if preds[i]==labels[i] else "red"))
        plt.axis("off")

        plt.subplot(2, num_samples, i + 1 + num_samples)
        plt.imshow(depth, cmap="inferno")
        plt.title("3D Depth (MiDaS)")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# 🔹 Run visual + 3D depth display
show_validation_predictions_with_3D(model, val_loader, dataset.classes)


In [ ]:
# ============================================================
# 🧩 U-Net Implementation for Image Segmentation (Easy Version)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os

# ============================================================
# ⚙️ U-Net Architecture
# ============================================================

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )

        self.down1 = conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.down4 = conv_block(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = conv_block(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = conv_block(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = conv_block(128, 64)

        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)
        d2 = self.down2(p1)
        p2 = self.pool2(d2)
        d3 = self.down3(p2)
        p3 = self.pool3(d3)
        d4 = self.down4(p3)
        p4 = self.pool4(d4)

        b = self.bottleneck(p4)

        up4 = self.up4(b)
        merge4 = torch.cat([up4, d4], dim=1)
        dec4 = self.dec4(merge4)
        up3 = self.up3(dec4)
        merge3 = torch.cat([up3, d3], dim=1)
        dec3 = self.dec3(merge3)
        up2 = self.up2(dec3)
        merge2 = torch.cat([up2, d2], dim=1)
        dec2 = self.dec2(merge2)
        up1 = self.up1(dec2)
        merge1 = torch.cat([up1, d1], dim=1)
        dec1 = self.dec1(merge1)
        out = self.final(dec1)

        return out


# ============================================================
# 🧩 Dummy Dataset for Example (You can replace with real data)
# ============================================================

class DummySegmentationDataset(Dataset):
    def __init__(self, size=100, image_size=128):
        self.size = size
        self.image_size = image_size
        self.transform = transforms.ToTensor()

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        img = Image.fromarray(np.uint8(np.random.rand(self.image_size, self.image_size, 3) * 255))
        mask = Image.fromarray(np.uint8(np.random.rand(self.image_size, self.image_size) > 0.5))
        return self.transform(img), self.transform(mask)

train_dataset = DummySegmentationDataset()
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# ============================================================
# ⚙️ Training Loop (Simple)
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs)
        loss = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

# ============================================================
# 🎨 Visualize Predictions
# ============================================================
model.eval()
imgs, masks = next(iter(train_loader))
with torch.no_grad():
    preds = torch.sigmoid(model(imgs.to(DEVICE))).cpu()
preds = (preds > 0.5).float()

plt.figure(figsize=(12, 6))
for i in range(3):
    plt.subplot(3, 3, i*3+1)
    plt.imshow(np.transpose(imgs[i], (1,2,0)))
    plt.title("Input")
    plt.axis("off")

    plt.subplot(3, 3, i*3+2)
    plt.imshow(masks[i][0], cmap="gray")
    plt.title("True Mask")
    plt.axis("off")

    plt.subplot(3, 3, i*3+3)
    plt.imshow(preds[i][0], cmap="gray")
    plt.title("Predicted Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 🧩 U-Net Segmentation + Auto-select Drive Image (Complete)
# Paste into Google Colab and run (one cell)
# ============================================================

# 0) Mount drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 1) imports
import os, glob
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# ---------------------------
# 2) U-Net definition
# ---------------------------
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )
        self.down1 = conv_block(in_channels, 64); self.pool1 = nn.MaxPool2d(2)
        self.down2 = conv_block(64, 128); self.pool2 = nn.MaxPool2d(2)
        self.down3 = conv_block(128, 256); self.pool3 = nn.MaxPool2d(2)
        self.down4 = conv_block(256, 512); self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = conv_block(512, 1024)
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2); self.dec4 = conv_block(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2); self.dec3 = conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2); self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2); self.dec1 = conv_block(128, 64)
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x); p1 = self.pool1(d1)
        d2 = self.down2(p1); p2 = self.pool2(d2)
        d3 = self.down3(p2); p3 = self.pool3(d3)
        d4 = self.down4(p3); p4 = self.pool4(d4)
        b = self.bottleneck(p4)
        up4 = self.up4(b); merge4 = torch.cat([up4, d4], dim=1); dec4 = self.dec4(merge4)
        up3 = self.up3(dec4); merge3 = torch.cat([up3, d3], dim=1); dec3 = self.dec3(merge3)
        up2 = self.up2(dec3); merge2 = torch.cat([up2, d2], dim=1); dec2 = self.dec2(merge2)
        up1 = self.up1(dec2); merge1 = torch.cat([up1, d1], dim=1); dec1 = self.dec1(merge1)
        out = self.final(dec1)
        return out

# ---------------------------
# 3) Dummy dataset (demo training)
# ---------------------------
from torchvision import transforms as T
class DummySegmentationDataset(Dataset):
    def __init__(self, size=100, image_size=128):
        self.size = size
        self.image_size = image_size
        self.transform = T.ToTensor()
    def __len__(self):
        return self.size
    def __getitem__(self, idx):
        img = Image.fromarray(np.uint8(np.random.rand(self.image_size, self.image_size, 3) * 255))
        mask = Image.fromarray(np.uint8((np.random.rand(self.image_size, self.image_size) > 0.5).astype('uint8')*255))
        return self.transform(img), self.transform(mask)

train_dataset = DummySegmentationDataset(size=200, image_size=128)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)

# ---------------------------
# 4) Train loop (small demo)
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device ->", DEVICE)
model = UNet(in_channels=3, out_channels=1).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for imgs, masks in train_loader:
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)
        preds = model(imgs)
        loss = criterion(preds, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    avg = total_loss/len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg:.4f}")

# ---------------------------
# 5) Find a real image in Drive automatically
# ---------------------------
# If you want to pick a single known file, set PICK_FIRST_IMAGE=False and set USER_IMAGE_PATH
PICK_FIRST_IMAGE = True
USER_IMAGE_PATH = ""  # e.g. "/content/drive/MyDrive/Images/my_img.png"

if not PICK_FIRST_IMAGE and USER_IMAGE_PATH:
    test_image_path = USER_IMAGE_PATH
else:
    # search common extensions in MyDrive (recursively)
    patterns = ["*.png","*.jpg","*.jpeg","*.bmp"]
    found = []
    base = "/content/drive/MyDrive"
    for p in patterns:
        found += glob.glob(os.path.join(base, "**", p), recursive=True)
    found = sorted(found)
    if len(found) == 0:
        # fallback: list root MyDrive content for debugging
        print("❌ No images found under /content/drive/MyDrive. Try uploading or placing images into Drive.")
        print("Try listing files here to debug:", os.listdir("/content/drive/MyDrive")[:50])
        raise FileNotFoundError("No images in Drive. Upload an image to /content/drive/MyDrive or set USER_IMAGE_PATH.")
    # pick first found image
    test_image_path = found[0]

print("✅ Using image:", test_image_path)

# ---------------------------
# 6) Preprocess & predict on selected image
# ---------------------------
# Load original (for display) and resized for model input
orig_img = Image.open(test_image_path).convert("RGB")
resize_to = (128,128)
transform = T.Compose([T.Resize(resize_to), T.ToTensor()])
input_img = transform(orig_img).unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    pred = torch.sigmoid(model(input_img)).cpu().squeeze(0)[0]   # HxW
pred_mask = (pred > 0.5).float().numpy()  # binary mask (H,W)

# Prepare overlay (resize original to match input for overlay)
orig_small = orig_img.resize(resize_to)
orig_arr = np.array(orig_small).astype(np.float32)/255.0  # H,W,3

# colored mask: green overlay
mask_color = np.zeros_like(orig_arr)
mask_color[...,1] = pred_mask  # green channel

alpha = 0.45
overlay = np.clip((1-alpha)*orig_arr + alpha*mask_color, 0, 1)

# ---------------------------
# 7) Display results (Input original, Binary mask, Overlay)
# ---------------------------
plt.figure(figsize=(14,5))

plt.subplot(1,3,1)
plt.imshow(orig_small)
plt.title("Input (resized)")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(pred_mask, cmap='gray')
plt.title("Predicted Mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(overlay)
plt.title("Overlay (green mask)")
plt.axis("off")

plt.tight_layout()
plt.show()

# ---------------------------
# 8) Save outputs next to the image (optional)
# ---------------------------
out_dir = os.path.dirname(test_image_path)
np.save(os.path.join(out_dir, "pred_mask.npy"), pred_mask)
from imageio import imwrite
imwrite(os.path.join(out_dir, "pred_mask.png"), (pred_mask*255).astype('uint8'))
imwrite(os.path.join(out_dir, "overlay.png"), (overlay*255).astype('uint8'))
print("Saved pred_mask.png and overlay.png to:", out_dir)
